Hybrid Search using Pinecone and Langchain

In [6]:
!pip install --upgrade --quiet pinecone pinecone-text pinecone-notebooks

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [21]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec

index_name = "hybrid-search-langchain-pinecone"
pc = Pinecone(api_key=PINECONE_API_KEY)


#create index if it doesn't exist
existing_indexes = pc.list_indexes().names()

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384, #dimension of the dense vector embeddings
        metric='dotproduct',
        spec=ServerlessSpec(cloud='aws', region='us-east-1'),
    )

In [22]:
index=pc.Index(index_name)
index

d:\AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
#from dense vector


os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACE_API_KEY")

from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings


C:\Users\prash\AppData\Local\Temp\ipykernel_16120\1558677028.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
d:\AI\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prash\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [30]:
# for sparse vector
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prash\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [31]:
sentences = [
    "The cat is on the table.",
    "The dog is in the garden.",
    "The bird is flying in the sky."    
]

#tfidf values on these sentence
bm25_encoder.fit(sentences)

bm25_encoder.dump("bm25_values.json")

bm25_encoder = bm25_encoder.load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00,  7.67it/s]


In [32]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index
)

In [33]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000014086FA47D0>, index=<pinecone.db_data.index.Index object at 0x00000140D3BBBB00>)

In [34]:
retriever.add_texts(
    [
    "The cat is on the table.",
    "The dog is in the garden.",
    "The bird is flying in the sky."    
]
)

100%|██████████| 1/1 [00:04<00:00,  4.24s/it]


In [35]:
retriever.invoke("Where is the cat?")

[Document(metadata={'score': 0.570169032}, page_content='The cat is on the table.'),
 Document(metadata={'score': 0.096830368}, page_content='The dog is in the garden.'),
 Document(metadata={'score': 0.0809960365}, page_content='The bird is flying in the sky.')]

In [36]:
retriever.invoke("Where is the dog?")

[Document(metadata={'score': 0.47668606}, page_content='The dog is in the garden.'),
 Document(metadata={'score': 0.135584831}, page_content='The cat is on the table.'),
 Document(metadata={'score': 0.103325367}, page_content='The bird is flying in the sky.')]